# Data Ingestion and VectorDB Construction
Noam S

In [ ]:
%pip install langchain-text-splitters langchain-huggingface langchain-chroma langchain-community


In [ ]:
%pip install ipywidgets


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


In [ ]:
# Function to create a local VectorDB from extracted documents with proper chunking and metadata for FinRAG
def create_vector_db(extracted_documents, persist_directory="./chroma_db"):
    print("Starting the chunking process...")
    
    # Financial reports contain lots of numeric figures and tables.
    # Chunk size is set to 1000 characters, with 150 characters overlap to keep tables whole.
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150,
        separators=["\n\n", "\n", " ", ""],
        length_function=len
    )
    
    all_chunks = []
    all_metadatas = []
    
    for file_name, text in extracted_documents.items():
        # Split the text into manageable semantic pieces
        chunks = text_splitter.split_text(text)
        all_chunks.extend(chunks)
        
        # Enriched metadata tags to allow the agent to know the source file of each fact
        metadatas = [{"source": file_name} for _ in chunks]
        all_metadatas.extend(metadatas)
        
        print(f"Created {len(chunks)} chunks from source: {file_name}")

    print("\nInitializing local embedding model (HuggingFace sentence-transformers/all-MiniLM-L6-v2)...")
    # Using a fast, local embedding model that runs well on standard machines
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    
    print(f"Building local VectorDB at '{persist_directory}'...")
    # Create the local vector database
    vector_db = Chroma.from_texts(
        texts=all_chunks,
        embedding=embeddings,
        metadatas=all_metadatas,
        persist_directory=persist_directory,
        collection_metadata={"hnsw:space": "cosine"}
    )
    
    print("VectorDB created and saved successfully!")
    return vector_db
